In [25]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate


In [26]:
## loading model from dotenv

import os
from dotenv import load_dotenv,dotenv_values
load_dotenv()

m2 = os.getenv("MODEL_2")
print(m2)

lfm2.5-thinking:1.2b


In [27]:
#model = OllamaLLM(model="llama3.1:8b")    ##directly use model name without dotenv
model = OllamaLLM(model=m2)

In [28]:
template = """
You are an expert in recommendation for the restaurant named "Pizza Place" and thus will be asked about various type of restaurants and where the best type of particular food will be available, etc.
You might also be expected to plan some retaurants based on rough iternary and thu, you would be expected to recommend multiple restaurants at a time. 

Example:
User: " We are going to a bar and then would be going to a have dinner after it, 
can you recommend some bar that bar that ha good beer and BBQ restaurant?"
Expected Workflow: First search for the firt type of restaurant and then look for the other and recommend them."

If the user asks for a particular style of pizza/dish, search for the pizza or dish that fits the description and include its name in the result. 
If you cannot find the appropriate result, inform the user about it.

Example: 
User: Do they have any vegan pizza.
Agent-action: 1.) first search for the required pizza, in this case you got a result with name "tropical blast".
If you also obtain the name of the dish/ pizza, be sure to include the name of the pizza/dish in the final result.


You should not be overly formal but at the ame time, keep your answers simple and easy to understand as the users typically prefer short answer with a touch of familiarity,
as if they are having a conversation with friend.
Do not hallucinate and give at-most 3 recomendation with the information such a the place's name .

Here are some relevant reviews: {reviews}

Here is the question to answer: {questions}
"""
prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

##test
chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":"What is the best pizza place in town?"})

'<think> Okay, let\'s tackle this. The user is asking for the best pizza place in town. The example they gave was when someone asked about a bar and BBQ, the assistant first looked for the first type (bar) and then another type (BBQ), but in this case, the user specifically wants the best pizza place. \n\nWait, the user\'s query is "What is the best pizza place in town?" So I need to follow the workflow. Since the user wants the best pizza place, I should first search for pizza places. But the example they provided was when the user asked for a bar and BBQ, the assistant first looked for the first type (bar) then the second (BBQ). But here, since the user is directly asking for pizza, maybe I should start by searching for pizza restaurants. \n\nThe user mentioned if I can\'t find it, inform them. But since I need to recommend multiple restaurants, but the instruction says to give at most 3 recommendations. Wait, the user said "recommend multiple restaurants at a time" but the instructi

In [29]:
## Write in a python script
'''
while True:
    question = input("Ask your question (q to quit):")
    if question == "q":
        break
    
    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})
    
'''

'\nwhile True:\n    question = input("Ask your question (q to quit):")\n    if question == "q":\n        break\n\n    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})\n\n'

In [30]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
import os
import pandas as pd

In [31]:
dir = "data/csv/RRr.csv"
df = pd.read_csv(dir)

In [32]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:v1")

In [33]:
db_location = "./chrome_langchain_db"
add_documents = not os.path.exists(db_location)

if add_documents:
    documents = []
    ids = []
    
    for i, row in df.iterrows():
        document = Document(
            page_content = row["Title"] + " " + row["Review"],
            metadata = {"rating": row["Rating"], "date":row["Date"]},
            id= str(i)
        )
        ids.append(str(i))
        documents.append(document)
    
    
vector_store = Chroma(
    collection_name = "restaurant_reviews",
    persist_directory=db_location,
    embedding_function = embeddings
)
        
if add_documents:
    vector_store.add_documents(documents=documents,ids = ids)


In [34]:
retriever = vector_store.as_retriever(
    search_kwargs = {"k":5}
)

In [35]:
import re
regex_pattern = r"<think>[\s\S]*?<\/think>\n?\n?"


In [ ]:
question = "What is the best pizza available here?"

reviews = retriever.invoke(question)

result = chain.invoke({"reviews":reviews,"questions":question})
#cleaned_response = re.sub(regex_pattern, '', result)
print(result) # Output: This is the actual answer


The pepperoni pizza is the top pick! 🍕
